In [1]:
!head ratings_long.csv

userId,movieId,rating
0,16,5
0,72,5
0,86,5
0,259,1
0,319,4
0,521,4
0,534,2
0,671,1
0,673,2


In [3]:
import numpy as np
import pandas as pd

In [4]:
r = np.full((20, 1000),fill_value=np.nan)

In [5]:
df = pd.read_csv('ratings_long.csv')

In [6]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [7]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

In [18]:
def build_rating_matrix(file_path: str = "ratings_long.csv",
                        n_users: int = 20,
                        n_movies: int = 1000) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Builds a sparse user-movie rating matrix R from ratings_long.csv.

    R[userId, movieId] = rating
    Missing ratings are represented as np.nan.
    """
    df_local = pd.read_csv(file_path)

    required_columns = {"userId", "movieId", "rating"}
    missing_columns = required_columns - set(df_local.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    R = np.full((n_users, n_movies), fill_value=np.nan, dtype=float)

    for rec in df_local.itertuples(index=False):
        user_id = int(rec.userId)
        movie_id = int(rec.movieId)
        rating = float(rec.rating)

        if not (0 <= user_id < n_users):
            raise ValueError(f"userId out of range: {user_id}")

        if not (0 <= movie_id < n_movies):
            raise ValueError(f"movieId out of range: {movie_id}")

        R[user_id, movie_id] = rating

    return R, df_local


try:
    R = r.astype(float).copy()
    df = pd.read_csv("ratings_long.csv")
except NameError:
    R, df = build_rating_matrix("ratings_long.csv", n_users=20, n_movies=1000)


observed_mask = np.isfinite(R)
n_observed = int(observed_mask.sum())
sparsity = 1.0 - (n_observed / R.size)

print("Rating matrix shape:", R.shape)
print("Number of observed ratings:", n_observed)
print(f"Sparsity ratio: {sparsity:.4f}")

Rating matrix shape: (20, 1000)
Number of observed ratings: 200
Sparsity ratio: 0.9900


In [19]:
# Train / validation split

def make_train_validation_masks(mask: np.ndarray,
                                validation_ratio: float = 0.20,
                                random_state: int = 42) -> tuple[np.ndarray, np.ndarray]:
    """
    Splits observed entries into train and validation masks.
    Missing entries remain False in both masks.
    """
    rng = np.random.default_rng(random_state)

    observed_positions = np.argwhere(mask)
    rng.shuffle(observed_positions)

    n_total = len(observed_positions)
    n_valid = max(1, int(round(n_total * validation_ratio)))

    valid_positions = observed_positions[:n_valid]
    train_positions = observed_positions[n_valid:]

    train_mask = np.zeros_like(mask, dtype=bool)
    valid_mask = np.zeros_like(mask, dtype=bool)

    train_mask[train_positions[:, 0], train_positions[:, 1]] = True
    valid_mask[valid_positions[:, 0], valid_positions[:, 1]] = True

    return train_mask, valid_mask


train_mask, valid_mask = make_train_validation_masks(
    observed_mask,
    validation_ratio=0.20,
    random_state=42
)

print("Training ratings:", int(train_mask.sum()))
print("Validation ratings:", int(valid_mask.sum()))

Training ratings: 160
Validation ratings: 40


In [20]:
# Loss and metric functions 

def calculate_loss(U: np.ndarray,
                   V: np.ndarray,
                   R: np.ndarray,
                   mask: np.ndarray,
                   reg_lambda: float) -> float:
    """
    Computes regularized MSE loss only over observed entries.

    J(U,V) = 1/(2|Omega|) * sum((R_ij - U_i V_j)^2)
             + lambda/2 * (||U||_F^2 + ||V||_F^2)
    """
    predictions = U @ V
    errors = predictions[mask] - R[mask]

    data_loss = 0.5 * np.mean(errors ** 2)
    regularization_loss = 0.5 * reg_lambda * (
        np.sum(U ** 2) + np.sum(V ** 2)
    )

    return data_loss + regularization_loss


def calculate_metrics(U: np.ndarray,
                      V: np.ndarray,
                      R: np.ndarray,
                      mask: np.ndarray) -> tuple[float, float]:
    """
    Computes RMSE and MAE over selected observed entries.
    """
    if mask.sum() == 0:
        return np.nan, np.nan

    predictions = U @ V
    errors = predictions[mask] - R[mask]

    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))

    return rmse, mae

In [21]:
def train_matrix_factorization(R: np.ndarray,
                               train_mask: np.ndarray,
                               valid_mask: np.ndarray | None = None,
                               n_factors: int = 4,
                               learning_rate: float = 2.0,
                               reg_lambda: float = 0.01,
                               n_epochs: int = 3000,
                               print_every: int = 100,
                               patience: int = 600,
                               min_delta: float = 1e-6,
                               random_state: int = 42) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Learns U and V matrices such that:

        R ≈ U @ V

    using gradient descent over observed ratings only.
    """
    rng = np.random.default_rng(random_state)

    n_users, n_movies = R.shape

    
    U = 0.1 * rng.standard_normal((n_users, n_factors))
    V = 0.1 * rng.standard_normal((n_factors, n_movies))

    n_train = int(train_mask.sum())
    if n_train == 0:
        raise ValueError("Training mask has no observed ratings.")

    history = []

    best_score = np.inf
    best_U = U.copy()
    best_V = V.copy()
    epochs_without_improvement = 0

    for epoch in range(1, n_epochs + 1):
        predictions = U @ V

       
        E = np.zeros_like(R, dtype=float)
        E[train_mask] = predictions[train_mask] - R[train_mask]


        grad_U = (E @ V.T) / n_train + reg_lambda * U
        grad_V = (U.T @ E) / n_train + reg_lambda * V

     
        U -= learning_rate * grad_U
        V -= learning_rate * grad_V

        if epoch == 1 or epoch % print_every == 0 or epoch == n_epochs:
            train_loss = calculate_loss(U, V, R, train_mask, reg_lambda)
            train_rmse, train_mae = calculate_metrics(U, V, R, train_mask)

            if valid_mask is not None and valid_mask.sum() > 0:
                valid_rmse, valid_mae = calculate_metrics(U, V, R, valid_mask)
                score = valid_rmse
            else:
                valid_rmse, valid_mae = np.nan, np.nan
                score = train_rmse

            history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "train_rmse": train_rmse,
                "train_mae": train_mae,
                "valid_rmse": valid_rmse,
                "valid_mae": valid_mae
            })

            if score + min_delta < best_score:
                best_score = score
                best_U = U.copy()
                best_V = V.copy()
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += print_every

            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    history_df = pd.DataFrame(history)

    return best_U, best_V, history_df

In [22]:
U_validated, V_validated, validation_history = train_matrix_factorization(
    R=R,
    train_mask=train_mask,
    valid_mask=valid_mask,
    n_factors=4,
    learning_rate=2.0,
    reg_lambda=0.01,
    n_epochs=3000,
    print_every=100,
    patience=600,
    random_state=42
)

print("\nValidation experiment history:")
print(validation_history.tail().to_string(index=False))

train_rmse, train_mae = calculate_metrics(U_validated, V_validated, R, train_mask)
valid_rmse, valid_mae = calculate_metrics(U_validated, V_validated, R, valid_mask)

print("\nValidation experiment results")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Train MAE : {train_mae:.4f}")
print(f"Valid RMSE: {valid_rmse:.4f}")
print(f"Valid MAE : {valid_mae:.4f}")

Early stopping at epoch 600.

Validation experiment history:
 epoch  train_loss  train_rmse  train_mae  valid_rmse  valid_mae
   200    1.641782    0.557889   0.504867    3.311026   2.882736
   300    1.638701    0.557721   0.503779    3.297308   2.880235
   400    1.637069    0.557749   0.503070    3.288353   2.878995
   500    1.636102    0.557834   0.502574    3.281998   2.878124
   600    1.635488    0.557933   0.502217    3.277266   2.877437

Validation experiment results
Train RMSE: 3.3373
Train MAE : 3.0843
Valid RMSE: 3.2742
Valid MAE : 2.9532


In [23]:
u, v, final_history = train_matrix_factorization(
    R=R,
    train_mask=observed_mask,
    valid_mask=None,
    n_factors=4,
    learning_rate=2.0,
    reg_lambda=0.01,
    n_epochs=3000,
    print_every=100,
    patience=600,
    random_state=42
)

print("\nFinal training history:")
print(final_history.tail().to_string(index=False))

final_rmse, final_mae = calculate_metrics(u, v, R, observed_mask)

print("\nFinal model results on observed ratings")
print(f"Observed RMSE: {final_rmse:.4f}")
print(f"Observed MAE : {final_mae:.4f}")

Early stopping at epoch 1900.

Final training history:
 epoch  train_loss  train_rmse  train_mae  valid_rmse  valid_mae
  1500    1.781250    0.619890   0.551319         NaN        NaN
  1600    1.780956    0.619926   0.551116         NaN        NaN
  1700    1.780691    0.619968   0.550934         NaN        NaN
  1800    1.780453    0.620015   0.550769         NaN        NaN
  1900    1.780236    0.620065   0.550618         NaN        NaN

Final model results on observed ratings
Observed RMSE: 0.6199
Observed MAE : 0.5518


In [ ]:
# predictions

r_hat = u @ v


r_hat_clipped = np.clip(r_hat, 1.0, 5.0)

print("\nLearned matrix shapes")
print("u shape:", u.shape)
print("v shape:", v.shape)
print("r_hat shape:", r_hat.shape)

print("\nFirst 5x10 block of predicted rating matrix:")
print(pd.DataFrame(r_hat_clipped[:5, :10]).round(3))


assert u.shape == (R.shape[0], 4)
assert v.shape == (4, R.shape[1])
assert r_hat.shape == R.shape
assert np.isfinite(u).all()
assert np.isfinite(v).all()
assert np.isfinite(r_hat).all()

print("\nAll sanity checks passed.")


Learned matrix shapes
u shape: (20, 4)
v shape: (4, 1000)
r_hat shape: (20, 1000)

First 5x10 block of predicted rating matrix:
     0    1    2    3    4      5    6    7    8    9
0  1.0  1.0  1.0  1.0  1.0  1.000  1.0  1.0  1.0  1.0
1  1.0  1.0  1.0  1.0  1.0  1.000  1.0  1.0  1.0  1.0
2  1.0  1.0  1.0  1.0  1.0  1.000  1.0  1.0  1.0  1.0
3  1.0  1.0  1.0  1.0  1.0  2.345  1.0  1.0  1.0  1.0
4  1.0  1.0  1.0  1.0  1.0  1.998  1.0  1.0  1.0  1.0

All sanity checks passed.


In [27]:
train_positions = np.argwhere(train_mask)
valid_positions = np.argwhere(valid_mask)

train_users = set(train_positions[:, 0])
train_movies = set(train_positions[:, 1])

valid_users = set(valid_positions[:, 0])
valid_movies = set(valid_positions[:, 1])

cold_start_users = valid_users - train_users
cold_start_movies = valid_movies - train_movies

print("Validation users:", len(valid_users))
print("Validation movies:", len(valid_movies))

print("Cold-start users in validation:", len(cold_start_users))
print("Cold-start movies in validation:", len(cold_start_movies))

print("\nCold-start movie IDs:")
print(sorted(list(cold_start_movies))[:50])

print("\nValidation coverage ratio:")
print("Users covered by train:", 1 - len(cold_start_users) / max(1, len(valid_users)))
print("Movies covered by train:", 1 - len(cold_start_movies) / max(1, len(valid_movies)))

Validation users: 17
Validation movies: 38
Cold-start users in validation: 0
Cold-start movies in validation: 31

Cold-start movie IDs:
[np.int64(44), np.int64(59), np.int64(128), np.int64(140), np.int64(198), np.int64(323), np.int64(328), np.int64(332), np.int64(411), np.int64(428), np.int64(432), np.int64(445), np.int64(455), np.int64(457), np.int64(471), np.int64(497), np.int64(522), np.int64(543), np.int64(607), np.int64(611), np.int64(640), np.int64(666), np.int64(671), np.int64(679), np.int64(746), np.int64(779), np.int64(786), np.int64(791), np.int64(915), np.int64(942), np.int64(966)]

Validation coverage ratio:
Users covered by train: 1.0
Movies covered by train: 0.1842105263157895
